<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/foundry_local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundry Local

[Foundry Local](https://github.com/microsoft/Foundry-Local) from Microsoft exposes an OpenAI-compatible API for running local models. This example uses Foundry Local 2.2.1 through LlamaIndex's `OpenAILike` integration.

## Setup

1. Use Python 3.11-3.14 and install `llama-index-llms-openai-like` and `foundry-local-sdk==2.2.1` on Windows, macOS, or Linux. The unified SDK includes Windows support; uninstall the older `foundry-local-sdk-winml` package first if it is installed.
2. Use the Foundry Local SDK to download/load a model and start the optional local OpenAI-compatible web service.
3. `OpenAILike` needs an HTTP endpoint, so point it at `f"{manager.urls[0]}/v1"` and use `model.id` so requests target the exact loaded variant.
4. Foundry Local itself uses typed sessions for native in-process inference and does not require the web service outside OpenAI-compatible integrations. `OpenAILike` uses chat completions over HTTP, not the native Session API or the Responses API.
5. Increase `timeout` if your hardware takes longer to generate the first response.

Run this notebook on the machine that will host the model. In Google Colab, use a [local runtime](https://research.google.com/colaboratory/local-runtimes.html) to run on your own hardware.

In [ ]:
%pip install llama-index-llms-openai-like foundry-local-sdk==2.2.1

In [ ]:
from foundry_local_sdk import Configuration, FoundryLocalManager
from llama_index.llms.openai_like import OpenAILike

In [ ]:
FoundryLocalManager.initialize(
    Configuration(app_name="llamaindex-foundry-local")
)
manager = FoundryLocalManager.instance

model = manager.catalog.get_model("qwen2.5-0.5b")
model.download()
model.load()
manager.start_web_service()

llm = OpenAILike(
    model=model.id,
    api_base=f"{manager.urls[0].rstrip('/')}/v1",
    api_key="fake",
    context_window=32768,
    is_chat_model=True,
    is_function_calling_model=False,
    timeout=120.0,
)

In [ ]:
response = llm.complete("Reply with: Foundry Local is working.")
print(str(response))